In [1]:
### con FECHA ULTIMA ### Mejora --> 2 de febrero 2026

#!/usr/bin/env python3
"""
retry_keymetrics_ttm_failed.py

Reintento puntual de key-metrics TTM para tickers fallidos.
- Fecha manual opcional
- Fallback automático a última fecha en update_logs
- No mezcla fechas
- Secuencial, seguro y auditable
"""

import os
import time
import logging
import requests
from datetime import datetime, date, timedelta
from dotenv import load_dotenv
import psycopg2

# ===========================
# ENV
# ===========================
load_dotenv()

POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
API_KEY = os.getenv("FMP_API_KEY")

# ===========================
# CONFIG
# ===========================
FECHA_MANUAL = None
# Ejemplo:
# FECHA_MANUAL = date(2026, 2, 2)

# ===========================
# Logging
# ===========================
LOG_DIR = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = f"{LOG_DIR}/retry_keymetrics_ttm_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

logging.info("=== START retry_keymetrics_ttm_failed ===")

# ===========================
# DB
# ===========================
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        host=POSTGRES_HOST,
        port=POSTGRES_PORT
    )

# ===========================
# Obtener última fecha en logs
# ===========================
def obtener_ultima_fecha_logs(conn):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT MAX(created_at)::date
            FROM infraestructura.update_logs
            WHERE table_name = 'multifactor_keymetrics_ttm'
        """)
        r = cur.fetchone()
        return r[0]

# ===========================
# Obtener tickers fallidos
# ===========================
def obtener_tickers_fallidos(conn, fecha):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT ticker
            FROM infraestructura.update_logs
            WHERE table_name = 'multifactor_keymetrics_ttm'
              AND created_at >= %s
              AND created_at <  %s
              AND status != 'success'
        """, (fecha, fecha + timedelta(days=1)))
        return [r[0] for r in cur.fetchall()]

# ===========================
# API
# ===========================
def fetch_keymetrics_ttm(ticker):
    url = f"https://financialmodelingprep.com/api/v3/key-metrics-ttm/{ticker}?apikey={API_KEY}"
    r = requests.get(url, timeout=30)

    if r.status_code != 200:
        return None

    data = r.json()
    return data[0] if data else None

# ===========================
# SQL
# ===========================
INSERT_SQL = """
INSERT INTO api_raw.multifactor_keymetrics_ttm (
    ticker,
    fecha_de_consulta,
    freeCashFlowYieldTTM,
    marketCapTTM,
    netDebtToEBITDATTM,
    roicTTM
)
VALUES (
    %(ticker)s,
    %(fecha)s,
    %(freeCashFlowYieldTTM)s,
    %(marketCapTTM)s,
    %(netDebtToEBITDATTM)s,
    %(roicTTM)s
)
ON CONFLICT (ticker, fecha_de_consulta)
DO UPDATE SET
    freeCashFlowYieldTTM = EXCLUDED.freeCashFlowYieldTTM,
    marketCapTTM = EXCLUDED.marketCapTTM,
    netDebtToEBITDATTM = EXCLUDED.netDebtToEBITDATTM,
    roicTTM = EXCLUDED.roicTTM,
    updated_at = CURRENT_TIMESTAMP;
"""

# ===========================
# MAIN
# ===========================
def main():
    conn = get_conn()

    # ---------------------------
    # Resolver fecha objetivo
    # ---------------------------
    if FECHA_MANUAL:
        fecha_objetivo = FECHA_MANUAL
        logging.info(f"Usando fecha manual: {fecha_objetivo}")
    else:
        fecha_objetivo = obtener_ultima_fecha_logs(conn)
        if not fecha_objetivo:
            logging.error("No se pudo determinar fecha desde update_logs")
            conn.close()
            return
        logging.info(f"Usando última fecha detectada en logs: {fecha_objetivo}")

    # ---------------------------
    # Obtener tickers fallidos
    # ---------------------------
    tickers = obtener_tickers_fallidos(conn, fecha_objetivo)
    total = len(tickers)

    logging.info(f"Tickers a reintentar para {fecha_objetivo}: {total}")

    ok, fail = 0, 0

    with conn.cursor() as cur:
        for i, ticker in enumerate(tickers, 1):
            try:
                data = fetch_keymetrics_ttm(ticker)

                if not data:
                    fail += 1
                    logging.warning(f"{ticker} sigue sin datos")
                    continue

                payload = {
                    "ticker": ticker,
                    "fecha": fecha_objetivo,
                    "freeCashFlowYieldTTM": data.get("freeCashFlowYieldTTM"),
                    "marketCapTTM": data.get("marketCapTTM"),
                    "netDebtToEBITDATTM": data.get("netDebtToEBITDATTM"),
                    "roicTTM": data.get("roicTTM"),
                }

                cur.execute(INSERT_SQL, payload)
                conn.commit()

                ok += 1
                logging.info(f"{ticker} OK ({i}/{total})")

            except Exception as e:
                conn.rollback()
                fail += 1
                logging.error(f"{ticker} ERROR: {e}")

            time.sleep(0.4)

    conn.close()
    logging.info(f"FIN RETRY | FECHA={fecha_objetivo} | OK={ok} | FAIL={fail}")

# ===========================
if __name__ == "__main__":
    main()


2026-04-07 10:54:03,715 | INFO | === START retry_keymetrics_ttm_failed ===
2026-04-07 10:54:03,823 | INFO | Usando última fecha detectada en logs: 2026-04-07
2026-04-07 10:54:03,832 | INFO | Tickers a reintentar para 2026-04-07: 1
2026-04-07 10:54:04,566 | WARNING | CHAC sigue sin datos
2026-04-07 10:54:04,568 | INFO | FIN RETRY | FECHA=2026-04-07 | OK=0 | FAIL=1
